# LAB 4: Neural Sparse Search com SPLADE
## Implementação e Avaliação de Vetores Esparsos Neurais

**Objetivo:** Implementar Neural Sparse Search utilizando o modelo SPLADE (Sparse Lexical and Dense Embeddings) no OpenSearch.

**Contexto:** Implementaremos SPLADE localmente e indexaremos vetores esparsos com rank_features.

**Abordagem Prática:** Converter texto em vetores esparsos interpretáveis e integrar com buscas OpenSearch.

## Instalação de Dependências

In [ ]:
!pip install opensearch-py transformers torch numpy pandas matplotlib seaborn -q

## Importações

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from opensearchpy import OpenSearch
from transformers import AutoTokenizer, AutoModel
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Fundamentação Teórica: SPLADE

### O que é SPLADE?

SPLADE (Sparse Lexical and Dense Embeddings) é um modelo neuronal que produz vetores esparsos de alta dimensão, combinando eficiência léxica com significado semântico.

**Características principais:**

1. **Vetores Esparsos**: Apenas um pequeno subconjunto de dimensões possui valores não-zero
2. **Interpretabilidade**: Cada dimensão corresponde a um token; o peso representa a importância
3. **Expansão de Termos**: O modelo pode expandir sinônimos
4. **Hybrid Efficiency**: Combina velocidade de busca léxica com precisão semântica

### Arquitetura

SPLADE usa um encoder BERT base que aprende a:
- Selecionar tokens relevantes (sparsidade via ReLU)
- Atribuir pesos significativos (importância contextual)

### Vantagens

- **Velocidade**: Busca sparse é muito mais rápida que dense
- **Interpretabilidade**: Você vê quais termos impactam cada resultado
- **Eficiência**: Funciona bem em domínios especializados

## 2. Conexão com OpenSearch

In [ ]:
try:
    client = OpenSearch(
        hosts=[{'host': 'localhost', 'port': 9200}],
        http_auth=('admin', 'admin'),
        use_ssl=False,
        verify_certs=False,
        ssl_show_warn=False
    )
    info = client.info()
    print(f"Conectado ao OpenSearch {info['version']['number']}")
    client_available = True
except Exception as e:
    print(f"Aviso: OpenSearch não disponível ({e})")
    client_available = False
    client = None

## 3. Carregar Modelo SPLADE

In [ ]:
print("Carregando modelo SPLADE (pode levar alguns minutos)...")

model_name = "naver/splade-cocondenser-ensembledistil"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model = model.to(device)
    model.eval()
    print(f"Modelo SPLADE carregado: {model_name}")
    model_available = True
except Exception as e:
    print(f"Erro ao carregar modelo: {e}")
    print("Usando fallback com demonstração conceitual.")
    model_available = False
    tokenizer = None
    model = None

## 4. Implementar Função de Encoding SPLADE

In [ ]:
def encode_splade(text, tokenizer, model, device, top_k_tokens=20):
    """
    Codifica texto em vetor esparso SPLADE.
    """
    if not model_available or tokenizer is None:
        tokens_simulados = {text.split()[i % len(text.split())]: 0.5 + 0.3 * np.random.random() 
                           for i in range(min(top_k_tokens, len(text.split())))}
        return tokens_simulados
    
    try:
        tokens = tokenizer([text], padding=True, truncation=True, return_tensors='pt')
        tokens = {k: v.to(device) for k, v in tokens.items()}
        
        with torch.no_grad():
            outputs = model(**tokens)
        
        logits = outputs.logits
        weights = torch.relu(logits)
        weights_per_token = torch.max(weights, dim=1)[0][0].cpu().numpy()
        
        top_k_indices = np.argsort(weights_per_token)[-top_k_tokens:]
        
        result = {}
        for idx in reversed(top_k_indices):
            if weights_per_token[idx] > 0:
                token_str = tokenizer.decode([idx]).strip()
                result[token_str] = float(weights_per_token[idx])
        
        return result
    except Exception as e:
        print(f"Erro no encoding SPLADE: {e}")
        return {}


print("Função encode_splade definida.")

## 5. Exemplo: Codificar Termos Jurídicos

In [ ]:
termos_juridicos = [
    "prisão preventiva",
    "tráfico de drogas",
    "habeas corpus",
    "réu acusado imputado",
    "sentença condenatória"
]

print("=" * 80)
print("ANÁLISE SPLADE: TOP TOKENS PARA TERMOS JURÍDICOS")
print("=" * 80)

splade_encodings = {}

for termo in termos_juridicos:
    sparse_vec = encode_splade(termo, tokenizer, model, device, top_k_tokens=15)
    splade_encodings[termo] = sparse_vec
    
    print(f"\nTermo: {termo.upper()}")
    print("-" * 60)
    
    if sparse_vec:
        for token, weight in sorted(sparse_vec.items(), key=lambda x: x[1], reverse=True):
            bar = 'X' * int(weight * 20)
            print(f"  {token:20} | {weight:.4f} | {bar}")
    else:
        print("  (vetor vazio ou erro no processamento)")

## 6. Preparar Corpus Jurídico de Teste

In [ ]:
corpus_juridico = [
    {
        'id': 'doc_001',
        'titulo': 'Prisão Preventiva - Fundamentação Legal',
        'texto': 'A prisão preventiva é medida cautelar que pode ser decretada antes da condenação. Requer fundamentação baseada em risco ao resultado do processo ou aos bens jurídicos.'
    },
    {
        'id': 'doc_002',
        'titulo': 'Habeas Corpus - Proteção de Liberdade',
        'texto': 'Habeas corpus é remédio constitucional que protege o direito de liberdade. Pode ser impetrado contra coação ilegal de autoridade pública.'
    },
    {
        'id': 'doc_003',
        'titulo': 'Tráfico de Drogas - Pena e Procedimento',
        'texto': 'O tráfico de drogas é crime grave. A pena mínima é de cinco anos, podendo chegar a quinze anos conforme circunstâncias do delito.'
    },
    {
        'id': 'doc_004',
        'titulo': 'Acusado e Réu - Conceitos Processuais',
        'texto': 'Na linguagem jurídica, acusado é a pessoa física ou jurídica contra quem se propõe ação penal. Réu é a parte contrária no processo. Imputado refere-se àquele a quem se atribui crime.'
    },
    {
        'id': 'doc_005',
        'titulo': 'Sentença Condenatória - Natureza e Efeitos',
        'texto': 'Sentença condenatória é decisão judicial que acolhe a acusação e condena o réu. Produz efeitos civis, penais e processuais.'
    }
]

print(f"Corpus jurídico preparado com {len(corpus_juridico)} documentos")

## 7. Gerar Vetores Esparsos para Corpus

In [ ]:
corpus_with_sparse_vectors = []

print("Gerando vetores esparsos SPLADE para corpus...\n")

for doc in corpus_juridico:
    full_text = f"{doc['titulo']} {doc['texto']}"
    sparse_vec = encode_splade(full_text, tokenizer, model, device, top_k_tokens=30)
    
    doc['sparse_embedding'] = sparse_vec
    corpus_with_sparse_vectors.append(doc)
    
    print(f"Doc {doc['id']}: {len(sparse_vec)} tokens não-zero")

print(f"\nTodos os {len(corpus_with_sparse_vectors)} documentos processados.")

## 8. Criar Índice OpenSearch com rank_features

In [ ]:
index_name = 'neural_sparse_juridico'
index_mapping = {
    'settings': {
        'number_of_shards': 1,
        'number_of_replicas': 0
    },
    'mappings': {
        'properties': {
            'id': {'type': 'keyword'},
            'titulo': {'type': 'text'},
            'texto': {'type': 'text'},
            'sparse_embedding': {'type': 'rank_features'}
        }
    }
}

if client_available:
    try:
        try:
            client.indices.delete(index=index_name)
            print(f"Índice {index_name} deletado.")
        except:
            pass
        
        response = client.indices.create(index=index_name, body=index_mapping)
        print(f"Índice {index_name} criado com sucesso.")
    except Exception as e:
        print(f"Erro ao criar índice: {e}")
else:
    print("(OpenSearch não disponível - índice não será criado no servidor)")

## 9. Indexar Documentos

In [ ]:
if client_available:
    print(f"Indexando {len(corpus_with_sparse_vectors)} documentos...\n")
    
    for doc in corpus_with_sparse_vectors:
        doc_body = {
            'id': doc['id'],
            'titulo': doc['titulo'],
            'texto': doc['texto'],
            'sparse_embedding': doc['sparse_embedding']
        }
        
        try:
            response = client.index(index=index_name, id=doc['id'], body=doc_body)
            print(f"Indexado: {doc['id']} - {response['result']}")
        except Exception as e:
            print(f"Erro ao indexar {doc['id']}: {e}")
    
    try:
        client.indices.refresh(index=index_name)
        print(f"\nÍndice {index_name} atualizado.")
    except Exception as e:
        print(f"Erro ao refresh: {e}")
else:
    print("(OpenSearch não disponível - documentos não foram indexados)")

## 10. Implementar Busca com rank_features

In [ ]:
def search_neural_sparse(query_text, tokenizer, model, device, client=None, index_name='neural_sparse_juridico', k=5):
    """
    Buscar usando vetores esparsos SPLADE.
    """
    query_sparse_vec = encode_splade(query_text, tokenizer, model, device, top_k_tokens=50)
    
    if not query_sparse_vec:
        print("Aviso: vetor esparso da query está vazio")
        return []
    
    if client is None:
        print("(OpenSearch não disponível - usando busca local)")
        results = []
        for doc in corpus_with_sparse_vectors:
            doc_tokens = set(doc['sparse_embedding'].keys())
            query_tokens = set(query_sparse_vec.keys())
            overlap = len(doc_tokens & query_tokens)
            
            if overlap > 0:
                results.append({
                    '_id': doc['id'],
                    '_source': doc,
                    '_score': overlap / max(len(doc_tokens), len(query_tokens))
                })
        
        results = sorted(results, key=lambda x: x['_score'], reverse=True)[:k]
        return results
    
    try:
        search_body = {
            'size': k,
            'query': {
                'rank_features': {
                    'sparse_embedding': query_sparse_vec
                }
            }
        }
        
        response = client.search(index=index_name, body=search_body)
        return response['hits']['hits']
    except Exception as e:
        print(f"Erro em busca neural_sparse: {e}")
        return []


print("Função de busca neural_sparse definida.")

## 11. Teste de Busca Neural Sparse

In [ ]:
test_queries = [
    "prisão preventiva medida cautelar",
    "habeas corpus direito liberdade",
    "tráfico crime pena"
]

print("=" * 90)
print("BUSCA NEURAL SPARSE: RESULTADOS")
print("=" * 90)

search_results = {}

for query in test_queries:
    print(f"\nQuery: {query.upper()}")
    print("-" * 90)
    
    query_sparse = encode_splade(query, tokenizer, model, device, top_k_tokens=20)
    
    print(f"\nTokens mais relevantes da query:")
    for token, weight in sorted(query_sparse.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {token:20} {weight:.4f}")
    
    results = search_neural_sparse(query, tokenizer, model, device, client, index_name, k=5)
    search_results[query] = results
    
    print(f"\nResultados (top-5):")
    if results:
        for rank, result in enumerate(results, 1):
            doc_id = result['_id']
            score = result['_score']
            source = result['_source']
            titulo = source.get('titulo', 'N/A')
            print(f"  {rank}. [{doc_id}] {titulo} (score: {score:.4f})")
    else:
        print("  (sem resultados)")

## 12. Comparação Qualitativa: SPLADE vs BM25

In [ ]:
def search_bm25_local(query_text, corpus, k=5):
    """
    Busca BM25 simplificada (local, sem OpenSearch).
    """
    results = []
    query_words = set(query_text.lower().split())
    
    for doc in corpus:
        doc_words = set((doc['titulo'] + ' ' + doc['texto']).lower().split())
        overlap = len(query_words & doc_words)
        
        if overlap > 0:
            results.append({
                '_id': doc['id'],
                '_source': doc,
                '_score': overlap
            })
    
    results = sorted(results, key=lambda x: x['_score'], reverse=True)[:k]
    return results


print("=" * 90)
print("COMPARAÇÃO: NEURAL SPARSE (SPLADE) vs BM25")
print("=" * 90)

for query in test_queries:
    print(f"\nQuery: {query.upper()}")
    print("-" * 90)
    
    splade_results = search_neural_sparse(query, tokenizer, model, device, None, index_name, k=3)
    bm25_results = search_bm25_local(query, corpus_juridico, k=3)
    
    print(f"\n  SPLADE Results:")
    for rank, result in enumerate(splade_results, 1):
        titulo = result['_source'].get('titulo', 'N/A')
        print(f"    {rank}. {titulo}")
    
    print(f"\n  BM25 Results:")
    for rank, result in enumerate(bm25_results, 1):
        titulo = result['_source'].get('titulo', 'N/A')
        print(f"    {rank}. {titulo}")

## 13. Análise de Interpretabilidade: Tokens Mais Pesados

In [ ]:
termos_especificos = ["prisão", "tráfico", "habeas corpus"]

print("=" * 80)
print("INTERPRETABILIDADE: TOP TOKENS SPLADE POR TERMO JURÍDICO")
print("=" * 80)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, termo in enumerate(termos_especificos):
    sparse_vec = encode_splade(termo, tokenizer, model, device, top_k_tokens=12)
    
    tokens = list(sparse_vec.keys())
    weights = list(sparse_vec.values())
    
    ax = axes[idx]
    bars = ax.barh(range(len(tokens)), weights, color='steelblue', edgecolor='black')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.set_xlabel('Peso SPLADE', fontweight='bold')
    ax.set_title(f'Termo: {termo.upper()}', fontweight='bold')
    ax.invert_yaxis()
    
    for i, (bar, weight) in enumerate(zip(bars, weights)):
        ax.text(weight + 0.01, i, f'{weight:.3f}', va='center', fontsize=9)
    
    print(f"\nTermo: {termo.upper()}")
    print("-" * 60)
    for token, weight in sorted(sparse_vec.items(), key=lambda x: x[1], reverse=True):
        bar = 'X' * int(weight * 15)
        print(f"  {token:15} | {weight:.4f} | {bar}")

plt.tight_layout()
plt.savefig('splade_interpretability.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nGráfico salvo como 'splade_interpretability.png'")

## 14. Exercício: Expansão de Sinônimos Jurídicos

In [ ]:
sinonimos_juridicos = {
    'Acusado': ['réu', 'imputado', 'indiciado', 'acusado'],
    'Condenação': ['sentença condenatória', 'condenação', 'veredicto de culpa', 'sentença penal'],
    'Liberdade': ['liberdade de locomoção', 'direito de ir e vir', 'liberdade pessoal', 'liberdade']
}

print("=" * 90)
print("EXPANSÃO DE SINÔNIMOS: ANÁLISE SPLADE")
print("=" * 90)

for conceito, sinonimos in sinonimos_juridicos.items():
    print(f"\n### CONCEITO: {conceito.upper()} ###\n")
    
    sinonimo_vecs = {}
    for sin in sinonimos:
        vec = encode_splade(sin, tokenizer, model, device, top_k_tokens=8)
        sinonimo_vecs[sin] = vec
    
    all_tokens = set()
    for vec in sinonimo_vecs.values():
        all_tokens.update(vec.keys())
    
    print(f"Sinônimos analisados: {', '.join(sinonimos)}")
    print(f"Tokens únicos totais: {len(all_tokens)}")
    print(f"\nSobreposição de tokens:")
    
    for sin in sinonimos:
        tokens = set(sinonimo_vecs[sin].keys())
        print(f"  {sin:25} -> {len(tokens):2} tokens: {', '.join(list(tokens)[:5])}...")

## 15. Resumo Comparativo de Estratégias

In [ ]:
comparacao_data = {
    'Aspecto': [
        'Velocidade de Busca',
        'Tamanho de Índice',
        'Interpretabilidade',
        'Qualidade Semântica',
        'Eficiência Memória',
        'Aptidão para Domínios Especializados',
        'Capacidade de Expansão de Termos'
    ],
    'BM25': ['5/5', '3/5', '5/5', '2/5', '5/5', '3/5', '2/5'],
    'KNN Dense': ['2/5', '1/5', '1/5', '5/5', '2/5', '4/5', '4/5'],
    'SPLADE': ['4/5', '3/5', '5/5', '4/5', '4/5', '5/5', '5/5']
}

df_comparacao = pd.DataFrame(comparacao_data)

print("\n" + "=" * 100)
print("COMPARAÇÃO: BM25 vs KNN vs SPLADE")
print("=" * 100)
print(df_comparacao.to_string(index=False))
print("\nLegenda: Avaliação de 1/5 (pior) a 5/5 (melhor)")

## Referências (ABNT)

FORMAL, C. et al. **SPLADE: Sparse Lexical and Dense Embeddings for Efficient Information Retrieval**. In: International Conference on Learning Representations (ICLR), 2021. arXiv preprint arXiv:2107.05720, 2021. Disponível em: https://arxiv.org/abs/2107.05720. Acesso em: 2026.

NAVER. **SPLADE Models Repository**. GitHub: naver/splade-cocondenser-ensembledistil, 2021. Disponível em: https://github.com/naver/splade. Acesso em: 2026.

OPENSEARCH. **Neural Sparse Search Plugin**. OpenSearch Documentation, 2024. Disponível em: https://opensearch.org/docs/latest/ml-commons-plugin/neural-search/. Acesso em: 2026.

DEVLIN, J.; CHANG, M.-W.; LEE, K.; TOUTANOVA, K. **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**. In: Proceedings of the 2019 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies (NAACL-HLT), 2019. p. 4171-4186.

VASWANI, A. et al. **Attention is All You Need**. In: Advances in Neural Information Processing Systems (NeurIPS), 2017. p. 5998-6008.